In [17]:
year_month = 202401

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 19, Finished, Available, Finished, False)

In [18]:
from pyspark.sql.functions import col, to_date, regexp_replace, trim
from pyspark.sql import functions as F

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 20, Finished, Available, Finished, False)

In [19]:
df_silver = spark.sql(f"""
    SELECT *
    FROM transparencia_lh.dbo.bronze_despesas
    WHERE ano_mes = {year_month}
""")

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 21, Finished, Available, Finished, False)

In [20]:
df_silver = df_silver.withColumn(
    'ano_e_mes_do_lancamento', 
    to_date(col('ano_e_mes_do_lancamento'), 'yyyy/MM')
)


StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 22, Finished, Available, Finished, False)

In [21]:
money_columns = {
    'valor_pago_r': 'valor_pago_reais',
    'valor_empenhado_r': 'valor_empenhado_reais',
    'valor_liquidado_r': 'valor_liquidado_reais',
    'valor_restos_a_pagar_inscritos_r': 'valor_restos_a_pagar_inscritos_reais',
    'valor_restos_a_pagar_cancelado_r': 'valor_restos_a_pagar_cancelado_reais',
    'valor_restos_a_pagar_pagos_r': 'valor_restos_a_pagar_pagos_reais'
}


for old_name, new_name in money_columns.items():
    
    df_silver = df_silver.withColumnRenamed(old_name, new_name)
    
    df_silver = df_silver.withColumn(
        new_name,
        regexp_replace(col(new_name), r"\.", "")
    ).withColumn(
        new_name,
        regexp_replace(col(new_name), ",", ".")
    ).withColumn(
        new_name,
        col(new_name).cast("double")
    )

# df_silver.printSchema()

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 23, Finished, Available, Finished, False)

In [22]:
total_rows = df_silver.count()

null_percent = df_silver.select([

    (
        F.count(
            F.when(F.col(c).isNull(), c)
        ) / total_rows * 100
    ).alias(c)

    for c in df_silver.columns
])

display(null_percent)

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 275c1958-cab0-4829-90e5-e481c80e35df)

codigo_autor_emenda:
93% NULL porque só se aplica
a despesas originadas de emendas parlamentares.



In [23]:
df_silver = (

    df_silver

    .fillna({
        "nome_orgao_subordinado": "Sem informação",
        "nome_gestao": "Sem informação"
    })

    .fillna({
        "codigo_gestao": -3
    })

)



StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 25, Finished, Available, Finished, False)

### Tratamento de Valores Nulos

Foi realizada uma análise exploratória da quantidade de valores nulos por coluna na camada Silver para identificar se a ausência de dados representava erro de ingestão ou comportamento esperado do dataset.

Colunas como `municipio`, `uf` e `codigo_autor_emenda` apresentaram alto percentual de valores nulos (acima de 90%). Após análise semântica, concluiu-se que esses valores ausentes representam características naturais do domínio dos dados e não inconsistências da pipeline.

Por esse motivo, optou-se por manter os valores `NULL` nessas colunas, preservando a semântica original do dataset e evitando introdução artificial de valores como `"Sem informação"` ou códigos sentinela desnecessários.

Já para colunas onde o próprio dataset utiliza convenções específicas para ausência de informação, foram mantidos os padrões existentes, como:
- `-3` para campos numéricos
- `"Sem informação"` para campos textuais

Essa abordagem preserva a integridade analítica dos dados e evita distorções futuras em métricas, agregações e modelagens analíticas.


In [24]:
budget_fields = [
    "valor_empenhado_reais",
    "valor_liquidado_reais",
    "valor_pago_reais",
    "valor_restos_a_pagar_inscritos_reais",
    "valor_restos_a_pagar_cancelado_reais",
    "valor_restos_a_pagar_pagos_reais"
]

negative_percentage = df_silver.select([

    (
        F.count(
            F.when(F.col(c) < 0, c)
        ) / total_rows
    ).alias(c)

    for c in budget_fields
])

display(negative_percentage)

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 62f1483a-f630-477b-b4cc-7260e12b7a2b)

In [25]:
display(df_silver.filter(
    F.col("valor_liquidado_reais") < 0
))

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9ce17a53-2e39-4a00-a454-58b1b6c4e3b1)

In [26]:
display(df_silver.filter(
    F.col("valor_pago_reais") < 0
))

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6192476d-84ad-45db-a217-1453a75e58c2)

In [27]:
display(df_silver.filter(
    F.col("valor_restos_a_pagar_inscritos_reais") < 0
))

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9fbba014-a4bf-4a34-a1f0-27df1985b417)

In [28]:
display(df_silver.filter(
    F.col("valor_restos_a_pagar_inscritos_reais") < 0
))

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 674f06ca-6302-4e21-a324-197c4dfbf2c7)

In [29]:
df_silver = df_silver.withColumnRenamed(
    'codigo_subfucao',
    'codigo_subfuncao'
)

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 31, Finished, Available, Finished, False)

In [30]:
df_silver = df_silver.dropDuplicates()

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 32, Finished, Available, Finished, False)

In [31]:
df_silver = df_silver.dropna(how="all")

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 33, Finished, Available, Finished, False)

In [32]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ano_mes")
    .option(
        "replaceWhere",
        f"ano_mes = {year_month}"
    )
    .saveAsTable("silver_despesas")
)

StatementMeta(, 78525562-27bd-41fc-8c95-b4ab9ed7ac04, 34, Finished, Available, Finished, False)